In [10]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import torch.nn as nn
from torchinfo import summary
import statistics
import csv

In [11]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")
device = "cpu"

In [12]:
all = []

with open('embedding_dataset.csv', 'r') as csvfile:
    reader = csv.reader(csvfile)
    next(reader) #skips first line of csv with headers
    for line in reader:
        all.append(line)
        # match line[0]:
        #     case 'Pinaceae':
        #         pin.append(line)
        #         all.append(line)
        #     case 'Betulaceae':
        #         bet.append(line)
        #         all.append(line)
        #     case 'Cupressaceae':
        #         cup.append(line)
        #         all.append(line)
        #     case 'Sapindaceae':
        #         sap.append(line)
        #         all.append(line)
        #     case 'Fagaceae':
        #         fag.append(line)
        #         all.append(line)
        #     case _:
        #         print('unplanned classification')

In [13]:
length = len(all)

In [14]:
genus_toint_dict = {}
species_toint_dict = {}
family_toint_dict = {}
genus_tostring_dict = {}
species_tostring_dict = {}
family_tostring_dict = {}

fencode = 0
gencode = 0
sencode = 0

family = []
genus = []
species = []
year = []
embeddings_tensor = torch.zeros(length, 64)

for entry in all:
    if not entry[0] in family_toint_dict:
        family_toint_dict[entry[0]] = fencode
        family_tostring_dict[fencode] = entry[0]
        fencode += 1
    if not entry[1] in genus_toint_dict:
        genus_toint_dict[entry[1]] = gencode
        genus_tostring_dict[gencode] = entry[1]
        gencode += 1
    if not entry[2] in species_toint_dict:
        species_toint_dict[entry[2]] = sencode
        species_tostring_dict[sencode] = entry[2]
        sencode += 1

for i, entry in enumerate(all):
    family.append(family_toint_dict[entry[0]])
    genus.append(genus_toint_dict[entry[1]])
    species.append(species_toint_dict[entry[2]])
    year.append(int(entry[3]))
    embeddings_tensor[i] = torch.tensor(list(map(float, entry[5:])))

family_tensor = torch.tensor(family)
genus_tensor = torch.tensor(genus)
species_tensor = torch.tensor(species)
year_tensor = torch.tensor(year)

master = [family_tensor, genus_tensor, species_tensor, year_tensor, embeddings_tensor]

In [15]:
species_weight = np.array([0.0]*len(species_toint_dict))

for entry in master[2]:
    species_weight[entry] += 1.0

species_weight = torch.tensor(species_weight/length)

In [16]:
class embed_dataset(torch.utils.data.Dataset):
    def __init__(self, family, genus, species, year, embeddings):
        self.family = family
        self.genus = genus
        self.species = species
        self.year = year
        self.embeddings = embeddings

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        family = self.family[idx]
        genus = self.genus[idx]

        species = self.species[idx]
        label = torch.zeros(len(species_weight))
        label[species] = 1
        
        year = self.year[idx]
        embedding = self.embeddings[idx]
        return label, embedding


# Create dataset splits
# Set up dataloader for batches
batch_size = 1

train_dataset = embed_dataset(family_tensor, genus_tensor, species_tensor, year_tensor, embeddings_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [17]:
#define model parameters
class SimpleLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(64, 1024)
        self.l2 = nn.Linear(1024, 2048)
        self.l3 = nn.Linear(2048, 4096)
        self.l4 = nn.Linear(4096, 242)
        self.relu = nn.ReLU()

    def forward(self, embedding):
        x = self.relu(self.l1(embedding))
        x = self.relu(self.l2(x))
        x = self.relu(self.l3(x))
        return self.l4(x)

In [ ]:
import torch.optim as optim

learning_rate = 2e-6

model = SimpleLinear().to(device)

criterion = nn.CrossEntropyLoss(weight=species_weight)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

for epoch in range(10):
    model.train()
    total_loss = 0
    for label, embedding in train_dataloader:
        embedding, label = embedding.to(device), label.to(device)
        optimizer.zero_grad()
        logits = model(embedding)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} - Loss: {avg_loss:.6f}")

Epoch 1 - Loss: 0.000002
Epoch 2 - Loss: 0.000002
Epoch 3 - Loss: 0.000002
Epoch 4 - Loss: 0.000002
Epoch 5 - Loss: 0.000001
Epoch 6 - Loss: 0.000002
Epoch 7 - Loss: 0.000001
Epoch 8 - Loss: 0.000001
Epoch 9 - Loss: 0.000001
Epoch 10 - Loss: 0.000001
